## Referencing
In ERP research reference are typically designed to target a specific component while HMP tries to provide different components for a task or dataset. In the first approach it is often useful to choose a reference depending on where and what component you expect but when looking for a broad set of components this is most of the time not the best strategy as this will bias the signal towards a specific representation (same goes for most PCA/ICA decomposiiton or machine learning  approaches). Using an overall reference such as the common average or the REST reference is therefore recommended for HMP.

## Filtering
HMP being based on a cross-correlation of the signal and a pattern, oscillations faster than the pattern do not have much impact appart from having less nice event-related potential plots (thus usually a low-pass filter is only set for commodity), however large drifts in the signal can bias this cross-correlation. It is therefore important to control for these drifts without however inducing filtering artifact because of a too agressive filter. Therefore an offline high-pass (HP) filter needs to be carefully choosen for your data except in the rare case where your data is very clean. For signal with a lot of high amplitudes drift using a HP filter up to 0.5Hz can improve the speed and accuracy of the HMP estimates, nevertheless this HP filter value remains in the higher range thus a probable better default tradeoff between filtering artifacts and slow-drifts is to use a 0.1Hz HP filter as suggested by some authors (see [Acunzo et al.](https://doi.org/10.1016/j.jneumeth.2012.06.011) and [Tanner et al. 2015](https://doi.org/10.1111/psyp.12437)). Nevertheless a good defualt doesn't exist and will depend on the frequency composition of your signal and the time you'll model using HMP. Note that when using the filtering options in `hmp.io.preprocessing`, HMP uses the default MNE filter parameters.

For more information on filter see the dedicated MNE tutorials: [Filtering and resampling data](https://mne.tools/stable/auto_tutorials/preprocessing/30_filtering_resampling.html) and [Background information on filtering](https://mne.tools/stable/auto_tutorials/preprocessing/25_background_filtering.html). For HMP in the event where no HP is desired one can also center the epochs from baseline to the duration of the trial using the `center` parameter of either `hmp.basedata.default()` or in the method `crop_reject_epochs` of `hmp.basedata.BaseData`. Note though that this will also have negative consequences depending on the signal characteristics and the length of the durations to be modelled.

## Artifact handling
### Trial rejection
The **more trials the better HMP performs** even when the signal on some trials is not great. That being said very large artifacts can 1) be captured by the data reduction method (by default PCA), thus consuming a degree of freedom in the decomposition, and 2) be captured as an HMP event because of the exceedingly large likelihood gain of placing an event with the topography of that artifact. The impact of point 1 can be negligeable as long as enough PC have been selected to still contain useful information for HMP. Regarding point 2 this will depend on how many trials are in the modelled dataset, if enough trials are present to compensate, the large artefact will not have much influence on the results, if you have a lower trial count then this will be problematic. A useful strategy is to use the `reject_amplitude` parameter of either `hmp.basedata.default()` or in the method `crop_reject_epochs` of `hmp.basedata.BaseData`. Instead of rejecting a trial based on the whole epoch as done by most M/EEG softwares, HMP only looks if any electrode exceeds the value given in `reject_amplitude` in the interval between baseline and the duration of that trial (so for an epoch of 2 seconds, if modelling a duration up to the response that occured at 1.5 second, an artifact occuring between 1.5 and 2s will not be rejected). Thus a recommendation is to set a sensible `reject_amplitude` that does not remove too many trials while accounting for very large artifacts (e.g. artifacts exceeding 200 $\mu$V for typical lab EEG setting).

### Artifact correction
For artifact correction method, **ICA for blink removal is typically not very useful** because in most experimental design in cognitive neuroscience the participants blink after the response. In some case you might also want to actually keep the occasional trials where participant blinked during the trial and remove them based on the `reject_amplitude` parameter given above. If however you have very long trials during which participant randomly blink then you can apply such an ICA although usually blinks are only problematic if they systematically appear during some time after stimulus or before response.

Certain automated method can help in cleaning the signal to avoid trial rejection. For example the GEDAI algorithm is a pretty efficient and recent (thus still in development) technique to correct artifacts in the EEG (see the [documentation](https://neurotuning.github.io/gedai/latest/install/index.html) and the example 2 in the tutorial "Reading your own data". Applying this, or related, method prior to the HMP analyses can drastically reduce the trial loss induced by the rejection explained above. You should however keep in mind that in some cases, artifacts might relate to the fact that participants were not doing the task as expected (e.g. looking away or moving) and that such method might obscure these trials by correcting them.

## Relation between minimum duration and number of events
HMP uses a minimum duration between event peaks to avoid that the event collapse to the same time position (more on that in Tutorial 1). Typically this minimum duration, called **location** ($L$), is chosen as the width ($W$) of the event to be modelled (so 50ms by default). This however poses the strong constraint that the minimum duration ($T_{min}$) must be equal or larger to the number of events times the location $L$. Thus the minimum duration that you model needs to be carefully choosen as the maximum number of event $N$ HMP can fit will depend on it as:
$$
N = \frac{T_{min}}{L} + 1
$$
To illustrate if the minimum duration is 200ms and we use the defaults of HMP, we can fit up to 5 events (depending on exact sampling rate) in the data because of the 200ms contraint, independently of the mean duration. To relax that we can either modify $W$ or $L$ (but see tutorial 1) or we can 1) increase $T_{min}$ by discarding trials with durations deemed to be too short given the experimental design using the `min_duration` parameter of either `hmp.basedata.default()` or in the method `crop_reject_epochs` of `hmp.basedata.BaseData`.

## Number of components when using PCA

Overall the more the better but the more principal components you keep using the `n_comp` parameter, the more the RAM load is important and might exceed you personal computers capacity. A good trade-off is to requenst 99% of variance by specifying `n_conp=.99` but this is only reasonable if there are no large artifacts in the data (see previous section). Note also that adding more PCs than necessary can also increase noise and thus require a higher tolerance parameter (see Tutorial 1) instead of the `tolerance=1e-4` that is used by default.

## Sampling frequency of the signal

As for PCA the higher the number of samples in your signal the better. However increasing the smapling frequency will dramatically slow HMP and increase RAM usage. It is necessary to have a minimum sampling frequency of 100 ms if you're fitting the classical HMP model with a 50ms pattern width. If you wish to model the data using a smaller pattern then you'll need to increase sampling frequency depending on the new choosen length. A pattern defined by at least 5 samples is the minimum so if we want to model a 25 ms pattern then we need our signal to be at least sampled at 200Hz.
